In [1]:
# -*- coding: utf-8 -*-
import os
import re
import glob
import json
import warnings
import numpy as np
import pandas as pd

import xgboost as xgb
import joblib

# =========================
# 0) 全局：屏蔽警告（可选）
# =========================
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")

# =========================
# 1) 配置
# =========================
RANDOM_SEED = 42
THRESH = 0.5

FEATURE_FILE_GLOB = "../Malodors_Rule&FG&Morgan&StructKG_features.xlsx"
OPTUNA_RESULTS_CSV = "../optuna_singlemodel_XGB_30trials_5fold_results.csv"

OUT_MODEL_FILE = "./prediction_model/xgb_bestparams_fullfit.joblib"
OUT_PARAMS_JSON = "./prediction_model/xgb_bestparams_sanitized.json"

BASE_XGB_PARAMS = dict(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbosity=0,
    multi_strategy="multi_output_tree",  # 需要 xgboost>=2.0
)

# =========================
# 2) 版本检查
# =========================
def _ver_tuple(v: str):
    parts = re.split(r"[.+-]", v.strip())
    nums = []
    for p in parts[:3]:
        try:
            nums.append(int(p))
        except Exception:
            nums.append(0)
    while len(nums) < 3:
        nums.append(0)
    return tuple(nums)

def check_xgb_version():
    v = _ver_tuple(xgb.__version__)
    if v < (1, 6, 0):
        raise RuntimeError(f"xgboost=={xgb.__version__} 太旧：multi-label 需要 >=1.6")
    if "multi_strategy" in BASE_XGB_PARAMS and v < (2, 0, 0):
        raise RuntimeError(
            f"xgboost=={xgb.__version__} 不支持 multi_output_tree（需要>=2.0）。"
            f"升级或删掉 multi_strategy。"
        )

# =========================
# 3) 读取特征文件
# =========================
def find_feature_file(pattern: str):
    cands = sorted(glob.glob(pattern))
    if not cands:
        raise FileNotFoundError(
            f"找不到特征文件：{pattern}\n"
            f"请确认你保存的 xlsx 文件名，或修改 FEATURE_FILE_GLOB。"
        )
    return cands[-1]

def find_smiles_col(df: pd.DataFrame):
    cand = [c for c in df.columns if isinstance(c, str) and "smiles" in c.lower()]
    if not cand:
        return None
    for p in ["Canonical SMILES", "canonical_smiles", "SMILES", "smiles"]:
        for c in cand:
            if c.lower() == p.lower():
                return c
    return cand[0]

def is_binary_01_series(s: pd.Series) -> bool:
    if s.dtype == bool:
        return True
    if not np.issubdtype(s.dtype, np.number):
        return False
    vals = pd.unique(s.dropna())
    if len(vals) == 0:
        return False
    return set(vals).issubset({0, 1})

def infer_feature_cols(df: pd.DataFrame):
    """
    兼容：
    - 前缀列：Rule__/FG__/Morgan_/morgan_/fp_/bit_/KG_emb_ ...
    - 纯数字列：0..2047（Excel 读入后可能变 int 或 str）
    """
    cols = []
    for prefix in ["Rule__", "FG__", "Morgan_", "morgan_", "fp_", "ecfp_", "mfp_", "bit_", "KG_emb_"]:
        cols.extend([c for c in df.columns if isinstance(c, str) and c.startswith(prefix)])
    if cols:
        return list(cols)

    num_cols = []
    for c in df.columns:
        if isinstance(c, (int, np.integer)):
            num_cols.append(c)
        elif isinstance(c, str) and c.isdigit():
            num_cols.append(c)
    if num_cols:
        return sorted(num_cols, key=lambda x: int(x))

    raise ValueError("未找到特征列（Rule__/FG__/Morgan_/bit_/KG_emb_ 或 0..2047 纯数字列）。")

def infer_label_cols(df: pd.DataFrame, smiles_col: str, feature_cols: list):
    exclude = set(feature_cols)
    if smiles_col is not None:
        exclude.add(smiles_col)

    label_cols = []
    for c in df.columns:
        if c in exclude:
            continue
        if is_binary_01_series(df[c]):
            label_cols.append(c)

    if not label_cols:
        raise ValueError("未识别到标签列（0/1）。请确认文件中包含标签列。")
    return label_cols

def load_Xy_from_xlsx(xlsx_path: str):
    df = pd.read_excel(xlsx_path)

    # 固定按列位置划分：
    # 第 1 列：SMILES
    # 第 2–25 列：24 个气味描述符标签 y
    # 第 26 列及以后：特征 X
    smiles_col = df.columns[0]
    label_cols = list(df.columns[1:25])
    feature_cols = list(df.columns[25:])

    if len(label_cols) != 24:
        raise ValueError(
            f"标签列数量应为 24，但当前为 {len(label_cols)}，请检查数据列顺序。"
        )

    if len(feature_cols) == 0:
        raise ValueError("未找到特征列。请确认第 26 列及以后为特征列。")

    X = df.iloc[:, 25:].fillna(0).astype(np.float32).values
    y = df.iloc[:, 1:25].fillna(0).astype(int).values

    if X.shape[0] != y.shape[0]:
        raise ValueError("X 和 y 行数不一致，请检查文件。")

    return X, y, feature_cols, label_cols

# =========================
# 4) 读取最优参数（Optuna CSV）+ 类型修复
# =========================
def sanitize_xgb_params(params: dict) -> dict:
    p = dict(params)

    int_keys = ["n_estimators", "max_depth", "n_jobs", "random_state"]
    for k in int_keys:
        if k in p and p[k] is not None and not (isinstance(p[k], str) and p[k].strip() == ""):
            try:
                p[k] = int(float(p[k]))
            except Exception:
                pass

    float_keys = [
        "learning_rate", "subsample", "colsample_bytree",
        "min_child_weight", "reg_lambda", "reg_alpha", "gamma"
    ]
    for k in float_keys:
        if k in p and p[k] is not None and not (isinstance(p[k], str) and p[k].strip() == ""):
            try:
                p[k] = float(p[k])
            except Exception:
                pass

    # 一些可能是字符串的参数（例如 boosting / grow_policy 等）保持原样即可
    return p

def load_best_params_from_optuna_csv(csv_path: str):
    df = pd.read_csv(csv_path)
    if "AUPRC_macro" not in df.columns:
        raise ValueError(f"Optuna 结果文件缺少 AUPRC_macro 列：{csv_path}")

    best_row = df.sort_values("AUPRC_macro", ascending=False).iloc[0].to_dict()

    drop_cols = set([
        "trial", "AUPRC_macro",
        "AUROC_macro", "Accuracy_macro", "Precision_macro", "Recall_macro", "Specificity_macro",
    ])

    params = {}
    for k, v in best_row.items():
        if k in drop_cols:
            continue
        params[k] = v

    final_params = dict(BASE_XGB_PARAMS)
    final_params.update(params)
    final_params = sanitize_xgb_params(final_params)
    return final_params

In [3]:

check_xgb_version()

feature_file = find_feature_file(FEATURE_FILE_GLOB)
print("[INFO] Feature file:", feature_file)

X, y, feat_cols, label_cols = load_Xy_from_xlsx(feature_file)
print(f"[INFO] X={X.shape} | y={y.shape} | n_features={len(feat_cols)} | n_labels={len(label_cols)}")

best_params = load_best_params_from_optuna_csv(OPTUNA_RESULTS_CSV)
print("[INFO] Best params (sanitized):")
print(json.dumps(best_params, indent=2, ensure_ascii=False, default=str))

# 保存一份 json（方便复现/写论文）
with open(OUT_PARAMS_JSON, "w", encoding="utf-8") as f:
    json.dump(best_params, f, indent=2, ensure_ascii=False, default=str)
print("[SAVED] params json ->", OUT_PARAMS_JSON)




[INFO] Feature file: ../Malodors_Rule&FG&Morgan&StructKG_features.xlsx
[INFO] X=(3756, 2595) | y=(3756, 24) | n_features=2595 | n_labels=24
[INFO] Best params (sanitized):
{
  "objective": "binary:logistic",
  "eval_metric": "logloss",
  "tree_method": "hist",
  "n_jobs": -1,
  "random_state": 42,
  "verbosity": 0,
  "multi_strategy": "multi_output_tree",
  "n_estimators": 433,
  "max_depth": 7,
  "learning_rate": 0.0350057872293877,
  "subsample": 0.9947153135691092,
  "colsample_bytree": 0.7778835626400454,
  "min_child_weight": 1.0072775841844182,
  "reg_lambda": 3.4681854273849724,
  "reg_alpha": 6.955456414716767e-08,
  "gamma": 3.606069985094933
}
[SAVED] params json -> ./prediction_model/xgb_bestparams_sanitized.json


In [4]:
# 全量训练
clf = xgb.XGBClassifier(**best_params)
clf.fit(X, y)
print("[INFO] Full-fit training done.")

# 保存模型包（含列名、阈值等）
joblib.dump(
    {
        "model": clf,
        "params": best_params,
        "feature_cols": feat_cols,
        "label_cols": label_cols,
        "threshold": THRESH,
        "random_seed": RANDOM_SEED,
        "xgboost_version": xgb.__version__,
    },
    OUT_MODEL_FILE
)
print("[SAVED] model package ->", OUT_MODEL_FILE)


[INFO] Full-fit training done.
[SAVED] model package -> ./prediction_model/xgb_bestparams_fullfit.joblib


In [5]:
# -*- coding: utf-8 -*-
import os
import numpy as np
import pandas as pd

# =========================
# 你只需要改这里
# =========================
IN_FILE = "./laogang_ALL_features.xlsx"      # ✅ 单个待预测文件（第1列SMILES，第2列开始全特征）
OUT_DIR = "./laogang_to_predict__pred"       # 输出文件夹
SHEET_NAME = 0
TOPK = 8

# notebook里必须已有：
# clf, feat_cols, label_cols
assert "clf" in globals(), "需要先在notebook里有 clf"
assert "feat_cols" in globals(), "需要先在notebook里有 feat_cols（训练时特征列名列表）"
assert "label_cols" in globals(), "需要先在notebook里有 label_cols（训练时标签列名列表）"
THRESH = globals().get("THRESH", 0.5)


# =========================
# 工具函数
# =========================
def strict_check_feature_block(df: pd.DataFrame, feat_cols: list):
    """
    严格规则：
    - 第1列是 SMILES
    - 第2列开始是特征块
    - 特征块列名序列必须与 feat_cols 完全一致（数量/顺序/名字）
    """
    if df.shape[1] < 2:
        raise ValueError("文件列数不足：至少需要 SMILES + 特征列")

    file_feat_cols = list(df.columns[1:])  # 第2列开始
    if len(file_feat_cols) != len(feat_cols):
        raise ValueError(f"特征列数量不一致：file={len(file_feat_cols)} vs trained={len(feat_cols)}")

    if file_feat_cols != list(feat_cols):
        diffs = []
        for i, (a, b) in enumerate(zip(file_feat_cols, feat_cols)):
            if a != b:
                diffs.append((i, a, b))
                if len(diffs) >= 30:
                    break
        msg = "特征列不一致（显示前30个差异：idx, file_col, trained_col）:\n" + \
              "\n".join([f"{i}: {fa} != {tb}" for i, fa, tb in diffs])
        raise ValueError(msg)

def extract_positive_proba_safe(clf, p, n_labels: int):
    """
    更稳的正类概率提取：
    - list: 每个标签一个 (n,2)，用 clf.classes_ 找到 class=1 的列
    - ndarray (n,L,2): 取 [:,:,1]
    - ndarray (n,L): 直接返回
    """
    # list: per-label probs
    if isinstance(p, list):
        out = []
        cls_all = getattr(clf, "classes_", None)

        for k, pi in enumerate(p):
            pi = np.asarray(pi)
            if pi.ndim != 2:
                raise ValueError(f"list[{k}] prob shape invalid: {pi.shape}")

            if pi.shape[1] == 1:
                # 极端情况：只有一列
                out.append(pi[:, 0])
                continue

            # 尝试获取该标签的 classes
            classes_k = None
            if isinstance(cls_all, list) and k < len(cls_all):
                classes_k = list(cls_all[k])
            elif isinstance(cls_all, (np.ndarray, list, tuple)):
                classes_k = list(cls_all)

            # 找到 class=1 的列
            if classes_k is not None and 1 in classes_k:
                j = classes_k.index(1)
            else:
                # 兜底：默认第二列是正类
                j = 1

            out.append(pi[:, j])

        return np.vstack(out).T.astype(np.float32)

    # ndarray
    p = np.asarray(p)
    if p.ndim == 3 and p.shape[-1] == 2:
        return p[:, :, 1].astype(np.float32)
    if p.ndim == 2:
        # (n,L)
        return p.astype(np.float32)

    raise ValueError(f"无法解析 predict_proba 输出形状: {p.shape}")

def make_topk_table(proba_df: pd.DataFrame, smiles: pd.Series, topk=8):
    labels = proba_df.columns.to_list()
    P = proba_df.to_numpy()
    out_rows = []
    for i in range(P.shape[0]):
        idx = np.argsort(-P[i])[:topk]
        row = {"row": int(i + 1), "SMILES": str(smiles.iloc[i])}
        for t, j in enumerate(idx, start=1):
            row[f"top{t}_label"] = labels[j]
            row[f"top{t}_proba"] = float(P[i, j])
        out_rows.append(row)
    return pd.DataFrame(out_rows)

def quick_debug_stats(X: np.ndarray, proba_df: pd.DataFrame, thresh: float):
    stats = {}
    stats["X_shape"] = tuple(X.shape)
    stats["X_nonzero_ratio"] = float(np.count_nonzero(X) / X.size) if X.size else 0.0
    stats["X_mean_abs"] = float(np.mean(np.abs(X))) if X.size else 0.0

    P = proba_df.to_numpy()
    stats["prob_global_min"] = float(np.min(P)) if P.size else np.nan
    stats["prob_global_mean"] = float(np.mean(P)) if P.size else np.nan
    stats["prob_global_max"] = float(np.max(P)) if P.size else np.nan
    stats["count_prob_gt_thresh"] = int(np.sum(P >= thresh)) if P.size else 0
    stats["any_prob_gt_thresh"] = bool((P >= thresh).any()) if P.size else False

    # 每行 top1 概率分布
    if P.size:
        top1 = np.max(P, axis=1)
        stats["top1_min"] = float(np.min(top1))
        stats["top1_mean"] = float(np.mean(top1))
        stats["top1_max"] = float(np.max(top1))
    return stats


# =========================
# 主流程：单文件预测
# =========================
os.makedirs(OUT_DIR, exist_ok=True)

base = os.path.splitext(os.path.basename(IN_FILE))[0]
print("[PRED]", IN_FILE)

df = pd.read_excel(IN_FILE, sheet_name=SHEET_NAME)

# 1) 严格检查特征块（第1列SMILES，第2列开始全特征）
strict_check_feature_block(df, feat_cols)

# 2) 提取 SMILES + 特征
smiles = df.iloc[:, 0].astype(str)
Xdf = df.iloc[:, 1:].copy()

# 3) 转数值；任何 NaN 行删除
Xdf = Xdf.apply(pd.to_numeric, errors="coerce")
valid_mask = ~Xdf.isna().any(axis=1)

n_all = len(df)
n_ok = int(valid_mask.sum())
n_drop = n_all - n_ok
print(f"[INFO] rows: total={n_all}, kept={n_ok}, dropped(NaN)={n_drop}")

if n_ok == 0:
    raise RuntimeError("所有样本都有缺失值（NaN），无法预测。请先清洗特征。")

smiles_keep = smiles.loc[valid_mask].reset_index(drop=True)
X = Xdf.loc[valid_mask].to_numpy(dtype=np.float32)

# 4) 预测（更稳的正类概率提取）
p = clf.predict_proba(X)
y_prob = extract_positive_proba_safe(clf, p, n_labels=len(label_cols))

proba_df = pd.DataFrame(y_prob, columns=label_cols)

# 二值（阈值可能导致全0，所以你先看 proba）
bin_df = (proba_df.values >= THRESH).astype(int)
bin_df = pd.DataFrame(bin_df, columns=label_cols)

# 5) 输出（SMILES + 预测标签）
out_proba = pd.concat([smiles_keep.rename("SMILES"), proba_df], axis=1)
out_bin   = pd.concat([smiles_keep.rename("SMILES"), bin_df], axis=1)
topk_df   = make_topk_table(proba_df, smiles_keep, topk=TOPK)

# 被删除的 NaN 行
dropped_df = df.loc[~valid_mask].copy()

# 6) 诊断信息（强烈建议保留，定位“全0”）
dbg = quick_debug_stats(X, proba_df, THRESH)
print("[DEBUG] Global max prob:", dbg["prob_global_max"])
print("[DEBUG] Any prob >= THRESH?", dbg["any_prob_gt_thresh"], "| count:", dbg["count_prob_gt_thresh"])
print("[DEBUG] X nonzero ratio:", dbg["X_nonzero_ratio"], "| mean(|X|):", dbg["X_mean_abs"])
print("[DEBUG] top1 max:", dbg.get("top1_max", None), "| top1 mean:", dbg.get("top1_mean", None))

# 7) 保存
out_proba_path   = os.path.join(OUT_DIR, f"{base}_pred_proba.xlsx")
out_bin_path     = os.path.join(OUT_DIR, f"{base}_pred_binary_thr{THRESH}.xlsx")
out_topk_path    = os.path.join(OUT_DIR, f"{base}_pred_top{TOPK}.xlsx")
out_dropped_path = os.path.join(OUT_DIR, f"{base}_dropped_nan_rows.xlsx")
out_debug_path   = os.path.join(OUT_DIR, f"{base}_debug.txt")

out_proba.to_excel(out_proba_path, index=False)
out_bin.to_excel(out_bin_path, index=False)
topk_df.to_excel(out_topk_path, index=False)
if len(dropped_df) > 0:
    dropped_df.to_excel(out_dropped_path, index=False)

with open(out_debug_path, "w", encoding="utf-8") as f:
    for k, v in dbg.items():
        f.write(f"{k}: {v}\n")

print("[SAVED]", out_proba_path)
print("[SAVED]", out_bin_path)
print("[SAVED]", out_topk_path)
print("[SAVED]", out_debug_path)
if len(dropped_df) > 0:
    print("[SAVED]", out_dropped_path)

print("\n[OK] Done. Outputs in:", OUT_DIR)


[PRED] ./laogang_ALL_features.xlsx
[INFO] rows: total=143, kept=143, dropped(NaN)=0
[DEBUG] Global max prob: 0.9702453017234802
[DEBUG] Any prob >= THRESH? True | count: 149
[DEBUG] X nonzero ratio: 0.012485010172871444 | mean(|X|): 0.012606276199221611
[DEBUG] top1 max: 0.9702453017234802 | top1 mean: 0.6250922083854675
[SAVED] ./laogang_to_predict__pred/laogang_ALL_features_pred_proba.xlsx
[SAVED] ./laogang_to_predict__pred/laogang_ALL_features_pred_binary_thr0.5.xlsx
[SAVED] ./laogang_to_predict__pred/laogang_ALL_features_pred_top8.xlsx
[SAVED] ./laogang_to_predict__pred/laogang_ALL_features_debug.txt

[OK] Done. Outputs in: ./laogang_to_predict__pred


In [5]:
# -*- coding: utf-8 -*-
import os
import numpy as np
import pandas as pd

# =========================
# 你只需要改这里
# =========================
IN_FILE = "./垃圾处理设施成分SMILES_feature.xlsx"      # ✅ 单个待预测文件（第1列SMILES，第2列开始全特征）
OUT_DIR = "./垃圾处理设施成分_to_predict__pred"       # 输出文件夹
SHEET_NAME = 0
TOPK = 8

# notebook里必须已有：
# clf, feat_cols, label_cols
assert "clf" in globals(), "需要先在notebook里有 clf"
assert "feat_cols" in globals(), "需要先在notebook里有 feat_cols（训练时特征列名列表）"
assert "label_cols" in globals(), "需要先在notebook里有 label_cols（训练时标签列名列表）"
THRESH = globals().get("THRESH", 0.5)


# =========================
# 工具函数
# =========================
def strict_check_feature_block(df: pd.DataFrame, feat_cols: list):
    """
    严格规则：
    - 第1列是 SMILES
    - 第2列开始是特征块
    - 特征块列名序列必须与 feat_cols 完全一致（数量/顺序/名字）
    """
    if df.shape[1] < 2:
        raise ValueError("文件列数不足：至少需要 SMILES + 特征列")

    file_feat_cols = list(df.columns[1:])  # 第2列开始
    if len(file_feat_cols) != len(feat_cols):
        raise ValueError(f"特征列数量不一致：file={len(file_feat_cols)} vs trained={len(feat_cols)}")

    if file_feat_cols != list(feat_cols):
        diffs = []
        for i, (a, b) in enumerate(zip(file_feat_cols, feat_cols)):
            if a != b:
                diffs.append((i, a, b))
                if len(diffs) >= 30:
                    break
        msg = "特征列不一致（显示前30个差异：idx, file_col, trained_col）:\n" + \
              "\n".join([f"{i}: {fa} != {tb}" for i, fa, tb in diffs])
        raise ValueError(msg)

def extract_positive_proba_safe(clf, p, n_labels: int):
    """
    更稳的正类概率提取：
    - list: 每个标签一个 (n,2)，用 clf.classes_ 找到 class=1 的列
    - ndarray (n,L,2): 取 [:,:,1]
    - ndarray (n,L): 直接返回
    """
    # list: per-label probs
    if isinstance(p, list):
        out = []
        cls_all = getattr(clf, "classes_", None)

        for k, pi in enumerate(p):
            pi = np.asarray(pi)
            if pi.ndim != 2:
                raise ValueError(f"list[{k}] prob shape invalid: {pi.shape}")

            if pi.shape[1] == 1:
                # 极端情况：只有一列
                out.append(pi[:, 0])
                continue

            # 尝试获取该标签的 classes
            classes_k = None
            if isinstance(cls_all, list) and k < len(cls_all):
                classes_k = list(cls_all[k])
            elif isinstance(cls_all, (np.ndarray, list, tuple)):
                classes_k = list(cls_all)

            # 找到 class=1 的列
            if classes_k is not None and 1 in classes_k:
                j = classes_k.index(1)
            else:
                # 兜底：默认第二列是正类
                j = 1

            out.append(pi[:, j])

        return np.vstack(out).T.astype(np.float32)

    # ndarray
    p = np.asarray(p)
    if p.ndim == 3 and p.shape[-1] == 2:
        return p[:, :, 1].astype(np.float32)
    if p.ndim == 2:
        # (n,L)
        return p.astype(np.float32)

    raise ValueError(f"无法解析 predict_proba 输出形状: {p.shape}")

def make_topk_table(proba_df: pd.DataFrame, smiles: pd.Series, topk=8):
    labels = proba_df.columns.to_list()
    P = proba_df.to_numpy()
    out_rows = []
    for i in range(P.shape[0]):
        idx = np.argsort(-P[i])[:topk]
        row = {"row": int(i + 1), "SMILES": str(smiles.iloc[i])}
        for t, j in enumerate(idx, start=1):
            row[f"top{t}_label"] = labels[j]
            row[f"top{t}_proba"] = float(P[i, j])
        out_rows.append(row)
    return pd.DataFrame(out_rows)

def quick_debug_stats(X: np.ndarray, proba_df: pd.DataFrame, thresh: float):
    stats = {}
    stats["X_shape"] = tuple(X.shape)
    stats["X_nonzero_ratio"] = float(np.count_nonzero(X) / X.size) if X.size else 0.0
    stats["X_mean_abs"] = float(np.mean(np.abs(X))) if X.size else 0.0

    P = proba_df.to_numpy()
    stats["prob_global_min"] = float(np.min(P)) if P.size else np.nan
    stats["prob_global_mean"] = float(np.mean(P)) if P.size else np.nan
    stats["prob_global_max"] = float(np.max(P)) if P.size else np.nan
    stats["count_prob_gt_thresh"] = int(np.sum(P >= thresh)) if P.size else 0
    stats["any_prob_gt_thresh"] = bool((P >= thresh).any()) if P.size else False

    # 每行 top1 概率分布
    if P.size:
        top1 = np.max(P, axis=1)
        stats["top1_min"] = float(np.min(top1))
        stats["top1_mean"] = float(np.mean(top1))
        stats["top1_max"] = float(np.max(top1))
    return stats


# =========================
# 主流程：单文件预测
# =========================
os.makedirs(OUT_DIR, exist_ok=True)

base = os.path.splitext(os.path.basename(IN_FILE))[0]
print("[PRED]", IN_FILE)

df = pd.read_excel(IN_FILE, sheet_name=SHEET_NAME)

# 1) 严格检查特征块（第1列SMILES，第2列开始全特征）
strict_check_feature_block(df, feat_cols)

# 2) 提取 SMILES + 特征
smiles = df.iloc[:, 0].astype(str)
Xdf = df.iloc[:, 1:].copy()

# 3) 转数值；任何 NaN 行删除
Xdf = Xdf.apply(pd.to_numeric, errors="coerce")
valid_mask = ~Xdf.isna().any(axis=1)

n_all = len(df)
n_ok = int(valid_mask.sum())
n_drop = n_all - n_ok
print(f"[INFO] rows: total={n_all}, kept={n_ok}, dropped(NaN)={n_drop}")

if n_ok == 0:
    raise RuntimeError("所有样本都有缺失值（NaN），无法预测。请先清洗特征。")

smiles_keep = smiles.loc[valid_mask].reset_index(drop=True)
X = Xdf.loc[valid_mask].to_numpy(dtype=np.float32)

# 4) 预测（更稳的正类概率提取）
p = clf.predict_proba(X)
y_prob = extract_positive_proba_safe(clf, p, n_labels=len(label_cols))

proba_df = pd.DataFrame(y_prob, columns=label_cols)

# 二值（阈值可能导致全0，所以你先看 proba）
bin_df = (proba_df.values >= THRESH).astype(int)
bin_df = pd.DataFrame(bin_df, columns=label_cols)

# 5) 输出（SMILES + 预测标签）
out_proba = pd.concat([smiles_keep.rename("SMILES"), proba_df], axis=1)
out_bin   = pd.concat([smiles_keep.rename("SMILES"), bin_df], axis=1)
topk_df   = make_topk_table(proba_df, smiles_keep, topk=TOPK)

# 被删除的 NaN 行
dropped_df = df.loc[~valid_mask].copy()

# 6) 诊断信息（强烈建议保留，定位“全0”）
dbg = quick_debug_stats(X, proba_df, THRESH)
print("[DEBUG] Global max prob:", dbg["prob_global_max"])
print("[DEBUG] Any prob >= THRESH?", dbg["any_prob_gt_thresh"], "| count:", dbg["count_prob_gt_thresh"])
print("[DEBUG] X nonzero ratio:", dbg["X_nonzero_ratio"], "| mean(|X|):", dbg["X_mean_abs"])
print("[DEBUG] top1 max:", dbg.get("top1_max", None), "| top1 mean:", dbg.get("top1_mean", None))

# 7) 保存
out_proba_path   = os.path.join(OUT_DIR, f"{base}_pred_proba.xlsx")
out_bin_path     = os.path.join(OUT_DIR, f"{base}_pred_binary_thr{THRESH}.xlsx")
out_topk_path    = os.path.join(OUT_DIR, f"{base}_pred_top{TOPK}.xlsx")
out_dropped_path = os.path.join(OUT_DIR, f"{base}_dropped_nan_rows.xlsx")
out_debug_path   = os.path.join(OUT_DIR, f"{base}_debug.txt")

out_proba.to_excel(out_proba_path, index=False)
out_bin.to_excel(out_bin_path, index=False)
topk_df.to_excel(out_topk_path, index=False)
if len(dropped_df) > 0:
    dropped_df.to_excel(out_dropped_path, index=False)

with open(out_debug_path, "w", encoding="utf-8") as f:
    for k, v in dbg.items():
        f.write(f"{k}: {v}\n")

print("[SAVED]", out_proba_path)
print("[SAVED]", out_bin_path)
print("[SAVED]", out_topk_path)
print("[SAVED]", out_debug_path)
if len(dropped_df) > 0:
    print("[SAVED]", out_dropped_path)

print("\n[OK] Done. Outputs in:", OUT_DIR)

[PRED] ./垃圾处理设施成分SMILES_feature.xlsx
[INFO] rows: total=19, kept=19, dropped(NaN)=0
[DEBUG] Global max prob: 0.9677262306213379
[DEBUG] Any prob >= THRESH? True | count: 39
[DEBUG] X nonzero ratio: 0.08152184801554706 | mean(|X|): 0.0150491027161479
[DEBUG] top1 max: 0.9677262306213379 | top1 mean: 0.8183627128601074
[SAVED] ./垃圾处理设施成分_to_predict__pred/垃圾处理设施成分SMILES_feature_pred_proba.xlsx
[SAVED] ./垃圾处理设施成分_to_predict__pred/垃圾处理设施成分SMILES_feature_pred_binary_thr0.5.xlsx
[SAVED] ./垃圾处理设施成分_to_predict__pred/垃圾处理设施成分SMILES_feature_pred_top8.xlsx
[SAVED] ./垃圾处理设施成分_to_predict__pred/垃圾处理设施成分SMILES_feature_debug.txt

[OK] Done. Outputs in: ./垃圾处理设施成分_to_predict__pred


In [6]:
# -*- coding: utf-8 -*-
import os
import numpy as np
import pandas as pd

# =========================
# 你只需要改这里
# =========================
FILE_RULE_FG   = "./world_feature/World_rule&fg_features.xlsx"
FILE_MORGAN    = "./world_feature/World_morgan_features.xlsx"
FILE_STRUCTKG  = "./world_feature/World_structkg_features.xlsx"

OUT_DIR = "./World_to_predict__pred"
SHEET_NAME = 0
TOPK = 8

# notebook 里必须已有：
assert "clf" in globals(), "需要先在notebook里有 clf"
assert "feat_cols" in globals(), "需要先在notebook里有 feat_cols（训练时特征列名列表）"
assert "label_cols" in globals(), "需要先在notebook里有 label_cols（训练时标签列名列表）"
THRESH = globals().get("THRESH", 0.5)

# =========================
# 工具函数（尽量不改你之前的逻辑）
# =========================
def extract_positive_proba_safe(clf, p, n_labels: int):
    """更稳的正类概率提取（保持你之前版本）"""
    if isinstance(p, list):
        out = []
        cls_all = getattr(clf, "classes_", None)

        for k, pi in enumerate(p):
            pi = np.asarray(pi)
            if pi.ndim != 2:
                raise ValueError(f"list[{k}] prob shape invalid: {pi.shape}")

            if pi.shape[1] == 1:
                out.append(pi[:, 0])
                continue

            classes_k = None
            if isinstance(cls_all, list) and k < len(cls_all):
                classes_k = list(cls_all[k])
            elif isinstance(cls_all, (np.ndarray, list, tuple)):
                classes_k = list(cls_all)

            if classes_k is not None and 1 in classes_k:
                j = classes_k.index(1)
            else:
                j = 1

            out.append(pi[:, j])

        return np.vstack(out).T.astype(np.float32)

    p = np.asarray(p)
    if p.ndim == 3 and p.shape[-1] == 2:
        return p[:, :, 1].astype(np.float32)
    if p.ndim == 2:
        return p.astype(np.float32)

    raise ValueError(f"无法解析 predict_proba 输出形状: {p.shape}")

def make_topk_table(proba_df: pd.DataFrame, smiles: pd.Series, topk=8):
    labels = proba_df.columns.to_list()
    P = proba_df.to_numpy()
    out_rows = []
    for i in range(P.shape[0]):
        idx = np.argsort(-P[i])[:topk]
        row = {"row": int(i + 1), "SMILES": str(smiles.iloc[i])}
        for t, j in enumerate(idx, start=1):
            row[f"top{t}_label"] = labels[j]
            row[f"top{t}_proba"] = float(P[i, j])
        out_rows.append(row)
    return pd.DataFrame(out_rows)

def strict_check_feature_cols_exact(file_feat_cols, feat_cols):
    """
    严格规则：拼接后的特征列（名字+顺序）必须与训练时 feat_cols 完全一致
    """
    if len(file_feat_cols) != len(feat_cols):
        raise ValueError(f"特征列数量不一致：combined={len(file_feat_cols)} vs trained={len(feat_cols)}")

    if list(file_feat_cols) != list(feat_cols):
        diffs = []
        for i, (a, b) in enumerate(zip(file_feat_cols, feat_cols)):
            if a != b:
                diffs.append((i, a, b))
                if len(diffs) >= 40:
                    break
        msg = "特征列不一致（显示前40个差异：idx, combined_col, trained_col）:\n" + \
              "\n".join([f"{i}: {fa} != {tb}" for i, fa, tb in diffs])
        raise ValueError(msg)

def normalize_smiles_series(s: pd.Series) -> pd.Series:
    # 只做最轻量的字符串规范化，避免因为空格/None 导致“假不一致”
    return s.astype(str).fillna("").str.strip()

def quick_debug_stats(X: np.ndarray, proba_df: pd.DataFrame, thresh: float):
    stats = {}
    stats["X_shape"] = tuple(X.shape)
    stats["X_nonzero_ratio"] = float(np.count_nonzero(X) / X.size) if X.size else 0.0
    stats["X_mean_abs"] = float(np.mean(np.abs(X))) if X.size else 0.0

    P = proba_df.to_numpy()
    stats["prob_global_min"] = float(np.min(P)) if P.size else np.nan
    stats["prob_global_mean"] = float(np.mean(P)) if P.size else np.nan
    stats["prob_global_max"] = float(np.max(P)) if P.size else np.nan
    stats["count_prob_gt_thresh"] = int(np.sum(P >= thresh)) if P.size else 0
    stats["any_prob_gt_thresh"] = bool((P >= thresh).any()) if P.size else False

    if P.size:
        top1 = np.max(P, axis=1)
        stats["top1_min"] = float(np.min(top1))
        stats["top1_mean"] = float(np.mean(top1))
        stats["top1_max"] = float(np.max(top1))
    return stats

# =========================
# 主流程：三文件拼接 -> 预测 -> 保存
# =========================
os.makedirs(OUT_DIR, exist_ok=True)

print("[READ]", FILE_RULE_FG)
df1 = pd.read_excel(FILE_RULE_FG, sheet_name=SHEET_NAME)

print("[READ]", FILE_MORGAN)
df2 = pd.read_excel(FILE_MORGAN, sheet_name=SHEET_NAME)

print("[READ]", FILE_STRUCTKG)
df3 = pd.read_excel(FILE_STRUCTKG, sheet_name=SHEET_NAME)

# --- 基本列数检查 ---
for k, d in enumerate([df1, df2, df3], start=1):
    if d.shape[1] < 2:
        raise ValueError(f"文件{k}列数不足：至少需要 SMILES + 特征列")

# --- SMILES 一致性（按行严格一致；不一致就报错）---
s1 = normalize_smiles_series(df1.iloc[:, 0])
s2 = normalize_smiles_series(df2.iloc[:, 0])
s3 = normalize_smiles_series(df3.iloc[:, 0])

if not (len(s1) == len(s2) == len(s3)):
    raise ValueError(f"三文件行数不一致：len1={len(s1)}, len2={len(s2)}, len3={len(s3)}")

m12 = (s1.values == s2.values)
m13 = (s1.values == s3.values)
if not (m12.all() and m13.all()):
    bad = np.where(~(m12 & m13))[0][:10]
    example = "\n".join([f"row={i+1}: f1='{s1.iloc[i]}' | f2='{s2.iloc[i]}' | f3='{s3.iloc[i]}'" for i in bad])
    raise ValueError("三文件 SMILES 按行不一致（显示前10个不一致行）：\n" + example)

smiles = s1.reset_index(drop=True)

# --- 按顺序拼接特征：df1[1:] + df2[1:] + df3[1:] ---
X1 = df1.iloc[:, 1:].copy()
X2 = df2.iloc[:, 1:].copy()
X3 = df3.iloc[:, 1:].copy()

combined_feat_cols = list(X1.columns) + list(X2.columns) + list(X3.columns)

# 严格检查：拼接后的特征列必须与训练 feat_cols 完全一致
strict_check_feature_cols_exact(combined_feat_cols, feat_cols)

# 合并成一个特征 DataFrame（列顺序按拼接顺序）
Xdf = pd.concat([X1, X2, X3], axis=1)

# --- 转数值；任何 NaN 行删除 ---
Xdf = Xdf.apply(pd.to_numeric, errors="coerce")
valid_mask = ~Xdf.isna().any(axis=1)

n_all = len(Xdf)
n_ok = int(valid_mask.sum())
n_drop = n_all - n_ok
print(f"[INFO] rows: total={n_all}, kept={n_ok}, dropped(NaN)={n_drop}")

if n_ok == 0:
    raise RuntimeError("所有样本都有缺失值（NaN），无法预测。请先清洗特征。")

smiles_keep = smiles.loc[valid_mask].reset_index(drop=True)
X = Xdf.loc[valid_mask].to_numpy(dtype=np.float32)

# --- 预测 ---
p = clf.predict_proba(X)
y_prob = extract_positive_proba_safe(clf, p, n_labels=len(label_cols))

proba_df = pd.DataFrame(y_prob, columns=label_cols)
bin_df = (proba_df.values >= THRESH).astype(int)
bin_df = pd.DataFrame(bin_df, columns=label_cols)

out_proba = pd.concat([smiles_keep.rename("SMILES"), proba_df], axis=1)
out_bin   = pd.concat([smiles_keep.rename("SMILES"), bin_df], axis=1)
topk_df   = make_topk_table(proba_df, smiles_keep, topk=TOPK)

# 被删除的 NaN 行（把三文件原始信息也一起保存，方便你回溯哪个文件哪列缺失）
dropped_idx = np.where(~valid_mask.values)[0]
dropped_pack = pd.DataFrame({"SMILES": smiles.iloc[dropped_idx].values})
# 只保存缺失行的拼接特征（便于排查）
dropped_feat = Xdf.iloc[dropped_idx].copy()
dropped_df = pd.concat([dropped_pack.reset_index(drop=True), dropped_feat.reset_index(drop=True)], axis=1)

# --- 诊断信息（保留，方便你确认不是“全0/阈值太高”）---
dbg = quick_debug_stats(X, proba_df, THRESH)
print("[DEBUG] prob_global_max:", dbg["prob_global_max"], "| any>=THRESH:", dbg["any_prob_gt_thresh"])
print("[DEBUG] X_nonzero_ratio:", dbg["X_nonzero_ratio"], "| mean(|X|):", dbg["X_mean_abs"])

# --- 保存 ---
base = "World_combined_RuleFG_Morgan_StructKG"
out_proba_path   = os.path.join(OUT_DIR, f"{base}_pred_proba.xlsx")
out_bin_path     = os.path.join(OUT_DIR, f"{base}_pred_binary_thr{THRESH}.xlsx")
out_topk_path    = os.path.join(OUT_DIR, f"{base}_pred_top{TOPK}.xlsx")
out_dropped_path = os.path.join(OUT_DIR, f"{base}_dropped_nan_rows.xlsx")
out_debug_path   = os.path.join(OUT_DIR, f"{base}_debug.txt")

out_proba.to_excel(out_proba_path, index=False)
out_bin.to_excel(out_bin_path, index=False)
topk_df.to_excel(out_topk_path, index=False)
if len(dropped_df) > 0:
    dropped_df.to_excel(out_dropped_path, index=False)

with open(out_debug_path, "w", encoding="utf-8") as f:
    for k, v in dbg.items():
        f.write(f"{k}: {v}\n")

print("[SAVED]", out_proba_path)
print("[SAVED]", out_bin_path)
print("[SAVED]", out_topk_path)
print("[SAVED]", out_debug_path)
if len(dropped_df) > 0:
    print("[SAVED]", out_dropped_path)

print("\n[OK] Done. Outputs in:", OUT_DIR)


[READ] ./world_feature/World_rule&fg_features.xlsx
[READ] ./world_feature/World_morgan_features.xlsx
[READ] ./world_feature/World_structkg_features.xlsx


ValueError: 特征列不一致（显示前40个差异：idx, combined_col, trained_col）:
0: Rule__FG__sulfoxide_or_sulfonyl_like__m0001 != FG: Sulfone(–SO2–)
1: Rule__FG__6_SS_6__m0002 != FG: Disulfide(S–S)
2: Rule__FG__CX3_O_6_6__m0003 != FG: [CX3](=O)[#6][#6]
3: Rule__FG__NX3_H2_H1_NC_O__m0004 != FG: NX3-H2-H1-NC-O
4: Rule__FG__S_H1__m0005 != FG: Thiol(–SH)
5: Rule__REGEX__ammonia_exact__m0006 != ammonia exact
6: Rule__ATOM__Cl__ge1p0__m0007 != Atom count: Cl >= 1
7: Rule__ATOM__N__ge2p0__m0008 != Atom count: N >= 2
8: Rule__ATOM__N__ge3p0__m0009 != Atom count: N >= 3
9: Rule__ATOM__O__ge3p0__m0010 != Atom count: O >= 3
10: Rule__ATOM__S__ge1p0__m0011 != Atom count: S >= 1
11: Rule__ATOM__S__ge2p0__m0012 != Atom count: S >= 2
12: Rule__FG__C_O_OH__m0013 != FG: Carboxylic acid(–COOH)
13: Rule__LOGIC__C_O_OH__MolWt_110__m0014 != C(=O)[OH] && (MolWt < 110)
14: Rule__FG__c1ccc_O_cc1__m0015 != FG: c1ccc(O)cc1
15: Rule__FG__c_CX3H1_O__m0016 != FG: c[CX3H1](=O)
16: Rule__FG__CC_C_C_O_O__m0017 != FG: CC(C)C(=O)O
17: Ru